# mmWave Radar Sensing: Bedroom Human Full RT

This tutorial shows how to run full-scene ray tracing for a moving AMASS/SMPL human in the bedroom scene. It uses the public APIs to make two focused comparisons:

1. retrace the scene at every chirp versus update a coherent path bank, both at max_depth=2;
2. use the coherent path bank at max_depth=1, 2, and 4.

Each comparison shows the mean range profile and the first-frame range-Doppler map. The four configurations are illustrative rather than benchmark or validation settings.

In [ ]:
# Resolve the local src/ tree and keep Matplotlib's cache outside the repository.
import os
import sys
import tempfile
import time

from pathlib import Path

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src" / "mmWaveRadar").exists():
    repo_root = repo_root.parent
if not (repo_root / "src" / "mmWaveRadar").exists():
    raise RuntimeError(
        "Run this notebook from inside a HERMES source checkout, "
        "including the top-level demo/ directory."
    )
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

os.environ.setdefault(
    "MPLCONFIGDIR",
    str(Path(tempfile.gettempdir()) / "mmwave-radar-mpl"),
)

import matplotlib.pyplot as plt
import numpy as np

from mmWaveRadar import amass_to_smpl_motion_sequence
from mmWaveRadar.dsp import (
    plot_axis_image,
    range_doppler_map,
    range_fft,
    range_profile_from_cube,
)
from mmWaveRadar.materials import human_skin_material
from mmWaveRadar.radar import FMCWConfig, RadarHardware, RadarSensor
from mmWaveRadar.simulation import (
    MmWaveRadarSimulator,
    db_relative,
    select_torch_device,
)
from mmWaveRadar.targets import MeshTarget
from mmWaveRadar.tutorial_support import (
    load_bedroom_scene,
    plot_bedroom_scene_projection,
    plot_mesh_projection,
    radar_pose_in_front_of_chest,
    rigid_transform_mesh_sequence,
    time_window_mesh_sequence,
)

## Radar Configuration

A single co-located Tx/Rx channel keeps the full-RT tutorial tractable. The FMCW transmitter count is derived from the hardware definition so the two cannot drift out of sync.

In [ ]:
hardware = RadarHardware.from_positions(
    tx_positions=[[0.0, 0.0, 0.0]],
    rx_positions=[[0.0, 0.0, 0.0]],
    name="single-virtual-channel",
)
fmcw = FMCWConfig(
    carrier_frequency=60e9,
    slope=68e12,
    chirp_duration=58e-6,
    chirp_repetition_time=65e-6,
    sampling_frequency=4.5e6,
    num_adc_samples=225,
    num_chirps_per_frame=64,
    frame_period=50e-3,
    num_tx=hardware.num_tx,
)

# One frame is enough for both a range profile and a 64-chirp Doppler FFT.
num_frames = 1
simulation_duration_s = num_frames * fmcw.frame_period

print(f"Wavelength: {1e3 * fmcw.wavelength:.2f} mm")
print(f"Virtual channels: {hardware.num_virtual_channels}")
print(
    f"Run length: {num_frames} frame x {fmcw.num_chirps_per_frame} chirps "
    f"({simulation_duration_s:.3f} s radar interval)"
)

## AMASS Motion Mesh

The CMU walking sequence is included with the repository. SMPL model files must be obtained separately and placed under models/smpl_models/, or supplied with MMWAVE_SMPL_MODEL_DIR.

In [ ]:
dataset_root = Path(
    os.environ.get("MMWAVE_DATASET_ROOT", repo_root / "data")
).expanduser()
amass_npz_path = Path(
    os.environ.get(
        "MMWAVE_AMASS_NPZ",
        dataset_root / "AMASS" / "walking_poses_cmu_105_02.npz",
    )
).expanduser()
smpl_model_dir = Path(
    os.environ.get("MMWAVE_SMPL_MODEL_DIR", repo_root / "models" / "smpl_models")
).expanduser()

missing_paths = [
    path for path in (amass_npz_path, smpl_model_dir) if not path.exists()
]
if missing_paths:
    missing = "\n".join(f"  - {path}" for path in missing_paths)
    raise FileNotFoundError(
        "Full-RT.ipynb requires these AMASS/SMPL inputs:\n"
        f"{missing}\n"
        "Obtain the licensed SMPL model separately and set "
        "MMWAVE_SMPL_MODEL_DIR if it is not under models/smpl_models/."
    )

smpl_device = select_torch_device()
smpl_mesh_sequence = amass_to_smpl_motion_sequence(
    str(amass_npz_path),
    str(smpl_model_dir),
    model_type="smpl",
    device=smpl_device,
)

# Select the walking portion and center its three preview poses in the bedroom.
motion_start_time_s = 3.0
mesh_preview_duration_s = 3.0
preview_source_times_s = motion_start_time_s + np.linspace(
    0.0, mesh_preview_duration_s, 3
)
preview_vertices = np.concatenate([
    smpl_mesh_sequence.vertices_at(time_s)
    for time_s in preview_source_times_s
])
preview_xy_center = 0.5 * (
    preview_vertices[:, :2].min(axis=0) + preview_vertices[:, :2].max(axis=0)
)
mesh_translation = np.array([
    1.5 - preview_xy_center[0],
    -preview_xy_center[1],
    -preview_vertices[:, 2].min(),
])
raw_mesh_sequence = rigid_transform_mesh_sequence(
    smpl_mesh_sequence,
    translation=mesh_translation,
)

# Rebase the selected source window to radar time zero.
mesh_sequence = time_window_mesh_sequence(
    raw_mesh_sequence,
    start_time_s=motion_start_time_s,
    duration_s=max(simulation_duration_s, mesh_preview_duration_s),
)

radar_position, radar_orientation, chest_front_point, _ = (
    radar_pose_in_front_of_chest(
        mesh_sequence,
        clearance_m=0.50,
    )
)
radar = RadarSensor(
    name="radar",
    position=tuple(float(value) for value in radar_position),
    orientation=tuple(float(value) for value in radar_orientation),
    hardware=hardware,
    fmcw=fmcw,
)

human_material = human_skin_material("human-full-rt-skin")
human_material.scattering_coefficient = 0.35
last_chirp_time_s = fmcw.chirp_time(
    num_frames - 1,
    fmcw.num_chirps_per_frame - 1,
)
if last_chirp_time_s > float(mesh_sequence.times[-1]) + 1e-9:
    raise ValueError(
        f"Last chirp time {last_chirp_time_s:.6f} s exceeds the mesh "
        f"window ending at {float(mesh_sequence.times[-1]):.6f} s."
    )

print(f"Motion: {amass_npz_path}")
print(f"SMPL model: {smpl_model_dir} ({smpl_device})")
print(
    f"Mesh: {mesh_sequence.vertex_count} vertices, "
    f"{mesh_sequence.faces.shape[0]} faces"
)
print(f"Radar position: {np.array2string(np.asarray(radar.position), precision=3)}")

### Human Mesh in the Bedroom

The three columns show evenly spaced poses over the preview window. The top row is the x-y projection and the bottom row is the x-z projection.

The bedroom geometry and radio materials are defined by `BEDROOM_SCENE_XML` in `src/mmWaveRadar/tutorial_support.py`. Modify that shared definition to change the traced scene. When changing geometry, also update `BEDROOM_SCENE_BOXES` in the same file so this 2D preview remains consistent with the RT scene.

In [ ]:
mesh_preview_times_s = np.linspace(
    float(mesh_sequence.times[0]),
    min(mesh_preview_duration_s, float(mesh_sequence.times[-1])),
    3,
)

fig, axes = plt.subplots(2, 3, figsize=(14, 7), constrained_layout=True)
for column, time_s in enumerate(mesh_preview_times_s):
    plot_mesh_projection(
        axes[0, column],
        mesh_sequence,
        time_s,
        radar.position,
        f"Human mesh x-y, t={time_s:.3f} s",
        axes=(0, 1),
    )
    plot_bedroom_scene_projection(axes[0, column], axes=(0, 1))

    plot_mesh_projection(
        axes[1, column],
        mesh_sequence,
        time_s,
        radar.position,
        f"Human mesh x-z, t={time_s:.3f} s",
        axes=(0, 2),
    )
    plot_bedroom_scene_projection(axes[1, column], axes=(0, 2))

plt.show()

## Full-RT Configurations

The per-chirp baseline sets periodic_retrace_period_chirps to 1. A coherent-bank case traces at the beginning of this one-frame run, then updates path geometry and phase at every chirp. All cases use the same scene, seed, path budget, and material settings.

In [ ]:
rt_samples_per_src = 80_000
rt_max_num_paths_per_src = 80_000
rt_diffuse_reflection = True
rt_seed = 42
coherent_depths = (1, 2, 4)

case_specs = [
    {
        "key": "retrace_depth2",
        "label": "Retrace every chirp, depth 2",
        "mobility_mode": "rt_retrace",
        "max_depth": 2,
        "retrace_period_chirps": 1,
    },
    *[
        {
            "key": f"coherent_depth{depth}",
            "label": f"Coherent bank, depth {depth}",
            "mobility_mode": "rt_coherent_bank",
            "max_depth": depth,
            "retrace_period_chirps": fmcw.num_chirps_per_frame,
        }
        for depth in coherent_depths
    ],
]


def run_case(spec):
    # Use fresh objects so this cell can be rerun safely.
    scene = load_bedroom_scene(merge_shapes=False)
    run_target = MeshTarget(
        name="amass_human_full_rt",
        mesh_sequence=mesh_sequence,
        material=human_material,
    )
    simulator = MmWaveRadarSimulator(
        mobility_mode=spec["mobility_mode"],
        max_depth=spec["max_depth"],
        samples_per_src=rt_samples_per_src,
        max_num_paths_per_src=rt_max_num_paths_per_src,
        coupling_mode="unrestricted",
        diffuse_reflection=rt_diffuse_reflection,
        periodic_retrace=True,
        periodic_retrace_period_chirps=spec["retrace_period_chirps"],
        compute_backend="auto",
        compute_precision="float32",
        adc_compute_precision="float32",
        progress=True,
        seed=rt_seed,
    )
    started = time.perf_counter()
    cube = simulator.run(scene, radar, [run_target], num_frames=num_frames)
    elapsed_s = time.perf_counter() - started
    print(f"{spec['label']}: {elapsed_s:.2f} s")
    return {**spec, "cube": cube, "wall_time_s": elapsed_s}


results = {spec["key"]: run_case(spec) for spec in case_specs}

## Range and Range-Doppler Products

The processing helper uses the standard DSP APIs. Every line and map in a figure uses the same dB reference, so configuration-dependent amplitude changes remain visible.

In [ ]:
range_limit_m = 6.0
display_floor_db = -80.0


def extract_products(cube):
    adc = np.asarray(cube.adc)
    chirps = adc.reshape((-1, adc.shape[-2], adc.shape[-1]))
    range_cube, profile_ranges = range_fft(
        chirps, fmcw=fmcw, window="hann", nfft_mult=4
    )
    profile = range_profile_from_cube(
        range_cube, combine_antennas="sum_power", combine_chirps="mean"
    )
    rd_cube, rd_ranges, velocities = range_doppler_map(
        adc[0],
        fmcw=fmcw,
        num_tx=hardware.num_tx,
        win_range="hann",
        win_doppler="hann",
    )
    return profile, profile_ranges, np.sum(np.abs(rd_cube) ** 2, axis=-1), rd_ranges, velocities

for result in results.values():
    result["products"] = extract_products(result["cube"])


def plot_comparison(result_keys, title):
    selected = [results[key] for key in result_keys]
    profile_ranges = selected[0]["products"][1]
    profile_mask = profile_ranges <= range_limit_m
    profiles_db = db_relative(np.stack([
        result["products"][0][profile_mask] for result in selected
    ]))
    rd_ranges = selected[0]["products"][3]
    rd_mask = rd_ranges <= range_limit_m
    rd_maps_db = db_relative(np.stack([
        result["products"][2][:, rd_mask] for result in selected
    ]))

    fig, axes = plt.subplots(
        1,
        len(selected) + 1,
        figsize=(5 * (len(selected) + 1), 4.2),
        constrained_layout=True,
    )

    for result, profile_db in zip(selected, profiles_db):
        axes[0].plot(
            profile_ranges[profile_mask],
            profile_db,
            label=result["label"],
        )
    axes[0].set(
        xlim=(0, range_limit_m),
        ylim=(display_floor_db, 3),
        xlabel="range [m]",
        ylabel="power [dB, shared reference]",
        title="Mean range profile",
    )
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=8)

    rd_axes = axes[1:]
    for ax, result, rd_map_db in zip(rd_axes, selected, rd_maps_db):
        velocities = result["products"][4]
        image = plot_axis_image(
            rd_map_db,
            rd_ranges[rd_mask],
            velocities,
            ax=ax,
            xlabel="range [m]",
            ylabel="velocity [m/s]",
            title=result["label"],
            cmap="magma",
            vmin=display_floor_db,
            vmax=0,
        )
    fig.colorbar(image, ax=rd_axes, label="power [dB, shared reference]")
    fig.suptitle(title)
    plt.show()

### Retracing Every Chirp vs Coherent Bank at max_depth=2

In [ ]:
plot_comparison(
    ["retrace_depth2", "coherent_depth2"],
    "Full RT mobility strategy at max_depth=2",
)

### Coherent Bank at max_depth=1, 2, and 4

In [ ]:
plot_comparison(
    ["coherent_depth1", "coherent_depth2", "coherent_depth4"],
    "Coherent path bank by maximum interaction depth",
)